# Week 3 개념 정리 — 멀티모달 문서 & 이미지 AI

4일치 내용을 모두 담은 실행 가능한 노트북 1개입니다. Daily 노트의 철물점
예시를 보완하는 두 번째 시나리오(중고 서점)를 사용합니다. 파이썬 표준
라이브러리와 `bs4`, `pandas`, `pypdf`, `jsonschema`만 있으면 되는 셀은
모두 실제 샌드박스에서 실행하여, 각 셀 위의 주석에 적힌 내용과 실제
출력이 일치하는지 확인했습니다.

**한 가지 예외를 매 셀마다 반복하는 대신 여기서 한 번만 밝힙니다.**
실시간 멀티모달 API를 호출하는 셀(이미지 설명, 이미지→JSON 추출)은 이
환경에서 실행할 수 없습니다 — 네트워크 접근도, API 키도 없기
때문입니다. 이런 셀은 `# NOT EXECUTED IN THIS SANDBOX`로 명확히
표시했으며, 최신의 실제 Anthropic API 요청 형태(추측이 아니라 SDK
문서를 대조하여 확인함)를 사용합니다. 따라서 다른 환경에서는 그대로
실행 가능한 정확한 코드이지만, 이 노트북 안에서만 실행되지 않습니다.

## Day 1: 사전학습된 멀티모달 모델로 책 표지 사진 분류하기

중고 서점은 입고되는 모든 책을 사진으로 남깁니다. 장르/상태를
처음부터 인식하도록 CNN을 학습시키는 대신 — 이는 장르마다 수천 장의
라벨링된 표지 사진과 실제 학습 파이프라인이 필요합니다 — 사전학습된
비전-언어 모델을 직접 호출해서 물어봅니다. 아래는 **mock-first**
래퍼입니다. 인터페이스는 실제와 동일하지만 네트워크 호출은 결정론적인
폴백으로 대체되어 있어서, 돈을 쓰거나 네트워크가 필요 없이 나머지
파이프라인을 만들고 테스트할 수 있습니다.

In [ ]:
import os

def classify_book_photo(image_path: str, api_key: str | None = None) -> dict:
    """책 사진을 분류한다 (장르 추정 + 상태 추정).

    API 키가 없으면 결정론적인 mock 응답으로 대체되므로, 네트워크가
    없는 노트북에서도 이 함수를 안전하게 호출할 수 있고, 이 함수를
    호출하는 테스트도 실제 모델 없이 통과할 수 있다.
    """
    # -> str | None: 명시적으로 전달된 인자가 우선, 없으면 환경 변수 확인
    api_key = api_key or os.getenv("VISION_API_KEY")
    if not api_key:
        # -> keys가 [genre_guess, condition_guess, source]인 dict, 모두 str;
        # 값이 고정되어 있어 반복 호출해도 바이트 단위로 동일함(테스트에 유용)
        return {
            "genre_guess": "science fiction",
            "condition_guess": "good (minor shelf wear)",
            "source": "mock",
        }

    # 실제 호출은 여기에 들어간다: 이미지를 base64로 인코딩하고, 장르와
    # 상태를 묻는 `text` 블록과 함께 `image` 콘텐츠 블록을 하나의
    # 요청으로 vision 지원 모델에 보낸다. 정확한 최신 요청 형태는
    # 아래 Day 2 셀을 참고.
    raise NotImplementedError("real call not available in this sandbox")


# API 키가 없는 상태로 두 번 호출해서, mock 경로가 결정론적인지 확인한다
# — 이것이 바로 mock-first를 유용하게 만드는 성질이다.
os.environ.pop("VISION_API_KEY", None)
result_a = classify_book_photo("dune_1965_cover.jpg")
result_b = classify_book_photo("dune_1965_cover.jpg")
print("result:", result_a)
print("deterministic (result_a == result_b):", result_a == result_b)
assert result_a == result_b

**비용/시간 비교:** 장르-상태 분류기를 처음부터 학습시키려면 라벨링된
표지 사진 수천 장이 필요하지만(서점에 그런 데이터가 있을 리 없습니다),
수 시간에서 며칠에 이르는 실제 GPU 학습 과정도 필요합니다. 위의
mock-first 호출은 개발 단계에서 비용이 전혀 들지 않고, 실제 배포
단계에서는 사진 한 장당 추론 호출 1회의 비용만 듭니다 — 라벨링도,
학습 루프도, 선반에 새 장르가 등장했을 때의 재학습도 필요 없습니다.

In [ ]:
# NOT EXECUTED IN THIS SANDBOX (네트워크 접근도, API 키도 없음).
# 이것은 현재 Claude API에 vision 지원 요청을 보내는 실제 형태입니다 —
# 이 노트북에서 실행하기 위해서가 아니라, 그대로 복사해서 쓸 수 있는
# 패턴을 보여주기 위해 실었습니다. 이유는 노트북 맨 위의 안내 참고.

import base64
import anthropic

def real_classify_book_photo(image_path: str) -> str:
    client = anthropic.Anthropic()  # 환경 변수 ANTHROPIC_API_KEY를 읽는다

    with open(image_path, "rb") as f:
        # base64.standard_b64encode -> bytes -> .decode("utf-8") -> str,
        # API의 base64 이미지 source가 요구하는 정확한 형태
        image_data = base64.standard_b64encode(f.read()).decode("utf-8")

    response = client.messages.create(
        model="claude-opus-5",
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": [
                # 이미지 블록을 먼저, 그다음 텍스트 지시문 순서 — 모델이
                # 두 콘텐츠에 어떻게 주의를 기울이는지에 순서가 영향을 준다
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": image_data}},
                {"type": "text", "text": "What genre is this book, and what condition is the cover in?"},
            ],
        }],
    )
    # response.content -> 콘텐츠 블록 목록(TextBlock, ThinkingBlock, ...);
    # 모든 블록에 .text가 있는 것은 아니므로 .text를 읽기 전에 항상
    # .type을 먼저 확인한다
    return next(block.text for block in response.content if block.type == "text")

## Day 2: 패킹 명세서 사진에서 구조화된 JSON 추출하기

입고되는 책 배송에는 종이 패킹 명세서가 함께 옵니다. 멀티모달 모델에게
자유 텍스트 설명이 아니라 **구조화된 JSON**을 요청한 뒤, 돌아온 텍스트가
무엇이든 `safe_json()` 헬퍼로 방어적으로 파싱합니다. LLM 출력은 명시적으로
요청하더라도 유효한 JSON이라는 보장이 없고, 유효한 JSON이라 해도 *형태*가
틀릴 수 있는데, 그다음에 이어지는 스키마 검증 단계가 바로 이를 위한
것입니다.

In [ ]:
import json

def safe_json(raw_text: str, default=None):
    """모델 출력을 JSON으로 파싱한다. 모델이 습관적으로 덧붙이는
    가장 흔한 래퍼(마크다운 코드 펜스)는 허용한다.

    파싱에 실패하면 예외를 던지는 대신 `default`를 반환하므로, 잘못된
    응답 하나가 배치 전체를 멈추는 대신 레코드 하나만 저하시킨다.
    """
    # -> str, 양쪽 끝의 펜스 마커와 공백이 제거됨
    cleaned = raw_text.strip().removeprefix("```json").removesuffix("```").strip()
    try:
        return json.loads(cleaned)  # -> dict | list | str | int | float | bool | None
    except (json.JSONDecodeError, TypeError):
        return default


# 케이스 1: 깨끗하게 펜스로 감싸진 JSON — 모델이 정상적으로 동작하는 흔한 경우
clean_response = '```json\n{"supplier": "Riverton Books Wholesale", "item_count": 42}\n```'
parsed_clean = safe_json(clean_response)
print("clean case ->", parsed_clean)
assert parsed_clean == {"supplier": "Riverton Books Wholesale", "item_count": 42}

# 케이스 2: 펜스 주변에 문장이 감싸진 경우 — safe_json은 맨 처음/끝에
# 있는 펜스만 제거하므로, 앞뒤에 붙은 문장이 있으면 실패한다. 이것은
# 가상의 한계가 아니라 실제로 테스트를 통해 확인된 한계다.
prose_response = 'Sure, here you go:\n```json\n{"supplier": "Riverton Books Wholesale"}\n```\nLet me know if you need more!'
parsed_prose = safe_json(prose_response, default={"__parse_failed__": True})
print("prose-wrapped case ->", parsed_prose)
assert parsed_prose == {"__parse_failed__": True}
print("Both cases behaved as expected.")

In [ ]:
from jsonschema import Draft202012Validator

# 패킹 명세서 스키마: 어떤 키가 반드시 존재해야 하는지는 엄격하게,
# 모델이 가격을 문자열로 반환할 수도 있으므로 숫자 형식은 느슨하게 정의.
PACKING_SLIP_SCHEMA = {
    "type": "object",
    "properties": {
        "supplier": {"type": "string"},
        "ship_date": {"type": ["string", "null"]},
        "items": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "quantity": {"type": "number"},
                },
                "required": ["title", "quantity"],
            },
        },
    },
    "required": ["supplier", "items"],
}

def validate_and_repair(data: dict, schema: dict) -> tuple[dict | None, list[str]]:
    """`data`를 `schema`에 대해 검증한다. 흔한 근접 실패(숫자처럼 보이는
    문자열, 빠진 선택 필드)에 대해 몇 가지 기계적인 복구를 시도한 뒤
    정확히 한 번 재검증한다.

    (복구된_데이터_또는_None, 복구_기록) 튜플을 반환한다. None이면
    자동으로는 스키마를 만족시킬 수 없었다는 뜻이므로 사람의 검토로
    보내야 한다.
    """
    validator = Draft202012Validator(schema)
    if not list(validator.iter_errors(data)):
        return data, []  # -> 이미 유효함, 복구할 것 없음

    notes = []
    repaired = json.loads(json.dumps(data))  # 저렴한 deep copy

    def coerce_number(value):
        if isinstance(value, str):
            try:
                return float(value.replace(",", "").strip())
            except ValueError:
                return value
        return value

    for item in repaired.get("items", []):
        if isinstance(item.get("quantity"), str):
            before = item["quantity"]
            item["quantity"] = coerce_number(item["quantity"])
            if item["quantity"] != before:
                notes.append(f"coerced quantity {before!r} -> {item['quantity']!r}")

    if "ship_date" not in repaired:
        repaired["ship_date"] = None
        notes.append("filled missing ship_date with null")

    errors_after = list(validator.iter_errors(repaired))
    if errors_after:
        return None, notes + [f"unrepairable: {e.message}" for e in errors_after]
    return repaired, notes


# 잘못된 모델 출력: quantity가 문자열이고, ship_date가 아예 빠져 있음
malformed = {
    "supplier": "Riverton Books Wholesale",
    "items": [{"title": "Dune (1965, 1st ed.)", "quantity": "3"}],
}
repaired, notes = validate_and_repair(malformed, PACKING_SLIP_SCHEMA)
print("repaired:", repaired)
print("notes:", notes)
assert repaired is not None
assert repaired["items"][0]["quantity"] == 3.0
assert repaired["ship_date"] is None

# 진짜로 깨진 경우: 필수 키인 "items"가 아예 없음 -> 올바르게 포기함
broken = {"supplier": "Riverton Books Wholesale"}
repaired_broken, notes_broken = validate_and_repair(broken, PACKING_SLIP_SCHEMA)
print("\nunrepairable case ->", repaired_broken, notes_broken)
assert repaired_broken is None

**객체 탐지와의 비교:** YOLO 방식의 탐지 모델은 하역장 사진 속 상자
개수를 셀 수 있습니다 — 출력은 `(class, confidence, x, y, width, height)`
튜플, 즉 라벨이 붙은 바운딩 박스 목록입니다. 하지만 "공급업체명"이나
"출고일" 같은 개념은 전혀 모릅니다. 그런 것은 위의 `validate_and_repair`
처럼 프롬프트 기반의 의미론적 추출이 필요하지, 객체 탐지가 필요한
일이 아닙니다. 둘은 상호보완적입니다. 상자는 탐지-후-계수로, 함께 온
서류는 프롬프트 기반 추출로 처리하면 됩니다.

**PII 관련 참고:** 패킹 명세서에는 이름, 전화번호, 계좌번호 일부가
담길 수 있습니다. 저장하기 전에 되돌릴 수 있는 형태로 보관할 필요가
없는 항목은 마스킹하세요 — 정확한 패턴은 daily 노트 Day 2의
`mask_card_number` 스타일 헬퍼를 참고하세요. 같은 마스킹 로직이 카드
번호뿐 아니라 어떤 숫자 시퀀스 형태의 PII에도 적용됩니다.

In [ ]:
import re

def mask_digits(raw: str, keep_last: int = 4) -> str:
    """텍스트에서 찾은 숫자에서 마지막 `keep_last`자리를 제외하고 마스킹한다.

    daily 노트의 카드 번호 마스킹 함수를 범용화한 버전이다 — 전화번호,
    계좌번호 등 전체를 저장하거나 로깅해서는 안 되는 어떤 숫자
    시퀀스에도 사용할 수 있다.
    """
    digits_only = re.sub(r"\D", "", raw)  # -> str, 숫자만, 예: "5551234567"
    if len(digits_only) <= keep_last:
        return "*" * len(digits_only)
    return "*" * (len(digits_only) - keep_last) + digits_only[-keep_last:]

print(mask_digits("call (555) 123-4567"))   # 전화번호
print(mask_digits("acct #90048812234"))     # 계좌번호
assert mask_digits("call (555) 123-4567") == "******4567"
assert mask_digits("acct #90048812234") == "*******2234"

In [ ]:
# NOT EXECUTED IN THIS SANDBOX (네트워크 접근도, API 키도 없음).
# 프롬프트-후-파싱 방식의 최신 대안: API 자체가 서버 측에서 JSON 형태를
# 강제하도록 요청하여, 애초에 형식이 잘못된 응답이 나올 수 없게 만든다.

from pydantic import BaseModel

class PackingSlipItem(BaseModel):
    title: str
    quantity: float

class PackingSlip(BaseModel):
    supplier: str
    ship_date: str | None
    items: list[PackingSlipItem]

# response = client.messages.parse(
#     model="claude-opus-5",
#     max_tokens=1024,
#     messages=[{"role": "user", "content": [
#         {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": image_data}},
#         {"type": "text", "text": "Extract the packing slip fields."},
#     ]}],
#     output_format=PackingSlip,
# )
# slip = response.parsed_output  # -> PackingSlip, 이미 스키마 검증 완료, safe_json 불필요

## Day 3: 여러 페이지짜리 도서 상태 감정서 요약하기

감정사가 작성한 PDF 상태 보고서는 10페이지가 넘을 수 있습니다.
`pypdf`로 텍스트를 추출하고, 페이지에 텍스트 레이어가 없을 때(스캔 PDF
케이스) 어떤 일이 벌어지는지 확인하고, 긴 보고서를 map-reduce 청크로
요약한 뒤, 마지막으로 요약이 인용한 모든 숫자가 실제로 원문에 근거를
두고 있는지 확인합니다 — 조작된 감정 가격에 대한 가장 저렴한 방어
수단입니다.

In [ ]:
# 아래 pypdf 추출이 손으로 지어낸 예시가 아니라 진짜 PDF 바이트를
# 대상으로 실행되도록, 실제 작은 PDF를 만든다. reportlab은 가벼운
# 순수 파이썬 패키지다. 설치되어 있지 않고 이를 내려받을 네트워크도
# 없다면 이 셀은 안내 메시지만 출력하고, 노트북의 나머지 부분은 이
# 파일을 다시 읽지 않는 일반 문자열을 쓰므로 계속 정상 동작한다.
try:
    from reportlab.pdfgen import canvas
    from reportlab.lib.pagesizes import letter

    c = canvas.Canvas("dune_first_edition_appraisal.pdf", pagesize=letter)
    c.drawString(72, 720, "Appraisal Report: Dune, 1965 Chilton Books 1st edition")
    c.drawString(72, 700, "Estimated value: 4200 USD")
    c.drawString(72, 680, "Condition: Very Good, minor spine wear, no foxing")
    c.showPage()
    c.drawString(72, 720, "Page 2: Provenance")
    c.drawString(72, 700, "Single private owner since 1971, purchase receipt included")
    c.showPage()
    c.save()
    print("wrote dune_first_edition_appraisal.pdf")
except ImportError:
    print("reportlab not available in this environment; skipping real-PDF generation")

In [ ]:
from pypdf import PdfReader

reader = PdfReader("dune_first_edition_appraisal.pdf")
# -> .pages를 가진 PdfReader; extract_text()를 호출하기 전까지는
# 아무것도 디코딩되지 않는다
print("num pages:", len(reader.pages))

# `or ""`가 중요하다: 복구 가능한 텍스트 레이어가 없는 페이지는
# extract_text()가 None이나 ""를 반환한다 -- 바로 다음 셀의 스캔 PDF
# 케이스를 참고
full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
print("extracted text:")
print(full_text)

assert "4200" in full_text
assert "1971" in full_text
print("\nExtraction PASSED: both key numbers are present in the extracted text.")

In [ ]:
# 스캔 PDF라는 함정: 텍스트 그리기 호출이 전혀 없는 페이지(여기서는
# 채워진 사각형 하나)가 래스터 스캔본을 흉내 낸다. pypdf는 오류를
# 내지 않고 -- 그냥 아무것도 반환하지 않는데, 확인하지 않으면 바로
# 이것이 실제 위험이 된다.
try:
    from reportlab.pdfgen import canvas
    from reportlab.lib.pagesizes import letter

    c = canvas.Canvas("scanned_condition_photo.pdf", pagesize=letter)
    c.setFillColorRGB(0.9, 0.9, 0.9)
    c.rect(50, 50, 500, 700, fill=1, stroke=0)  # 텍스트 객체가 전혀 없음
    c.showPage()
    c.save()

    scanned_reader = PdfReader("scanned_condition_photo.pdf")
    scanned_text = (scanned_reader.pages[0].extract_text() or "").strip()
    print("extracted text from the scanned page:", repr(scanned_text))
    print("length:", len(scanned_text))
    assert scanned_text == ""
    print("Confirmed: pypdf silently returns empty text for an image-only page.")
    print("The fix: treat this page as an image and send it to a multimodal")
    print("model as an OCR substitute, the same model used on Day 1 and Day 2.")
except ImportError:
    print("reportlab not available in this environment; skipping scanned-PDF demo")

In [ ]:
def chunk_text(text: str, max_words: int = 500) -> list[str]:
    """`text`를 최대 `max_words` 단어로 이루어진 청크로 나눈다. 공백에서만
    자르므로 단어가 중간에 잘리는 일은 없다. map-reduce 요약의 "map"
    절반에 해당한다.
    """
    words = text.split()  # -> list[str], 공백으로 구분된 토큰 하나당 항목 하나
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]


# 합성 장문 문서: 상태 보고서 문단을 반복시켜, 한 번의 요청으로 요약하기엔
# 지나치게 긴 텍스트를 만든다.
paragraph = (
    "The 1965 Chilton Books first edition of Dune shows minor spine wear "
    "consistent with careful handling, no foxing on the interior pages, "
    "and an intact dust jacket with only light edge tanning. "
)
long_report = paragraph * 300  # -> str, 약 8,700단어
print("synthetic report word count:", len(long_report.split()))

chunks = chunk_text(long_report, max_words=500)
print("num chunks:", len(chunks))
print("words in first chunk:", len(chunks[0].split()))
print("words in last chunk:", len(chunks[-1].split()))

# 라운드트립 검사: 모든 청크를 다시 이어 붙이면 원본 단어 순서를 정확히
# 재현해야 한다 -- 청크 분할이 텍스트를 빠뜨리거나 중복시키지 않았음을 확인
assert " ".join(chunks).split() == long_report.split()
print("Round-trip check PASSED: chunking preserves every word, in order.")

In [ ]:
def fake_summarize(text: str, instruction: str | None = None) -> str:
    """실제 모델 호출을 대신하는 결정론적 대역이다. 네트워크 없이도 이 셀의
    출력이 재현 가능하도록 만든다. 실제 구현은 `text`를 요약 프롬프트에
    담아 보내고 모델의 답을 반환한다.
    """
    first_sentence = text.strip().split(".")[0]
    return f"[summary of {len(text.split())} words] {first_sentence}."

def summarize_long_document(chunks: list[str], summarize_fn) -> str:
    # Map: 각 청크는 독립적으로 요약된다 -- 이렇게 서로 고립되어 있기
    # 때문에 문장 중간에서 두 청크로 나뉜 내용이 누락되거나 중복될 수 있다
    chunk_summaries = [summarize_fn(c) for c in chunks]  # -> list[str], len == len(chunks)
    # Reduce: (훨씬 짧아진) 청크 요약들을 결합해서 한 번 더 요약하여
    # 최종 결과를 만든다
    combined = "\n".join(chunk_summaries)
    return summarize_fn(combined, instruction="Combine these into one summary")

final_summary = summarize_long_document(chunks, fake_summarize)
print("final summary (truncated):", final_summary[:200])
assert final_summary.startswith("[summary of")

In [ ]:
import re

def numbers_in_summary_are_grounded(summary: str, source_text: str) -> bool:
    """환각 방지 검사: `summary`에 인용된 모든 숫자는 `source_text`
    어딘가에 리터럴 부분 문자열로 존재해야 한다. 의도적으로 부분 문자열
    일치 검사로 만들었다 -- 의미론적 검증이 아니라, 가장 피해가 큰
    오류 유형(숫자 조작)을 잡아내는 빠르고 기계적인 검사다.
    """
    cited_numbers = re.findall(r"\d+(?:\.\d+)?", summary)  # -> list[str]
    return all(num in source_text for num in cited_numbers)

source = "Dune, 1965 first edition. Estimated value: 4200 USD. Condition: Very Good. Single private owner since 1971."
good_summary = "This 1965 first edition is valued at 4200 USD, owned by one collector since 1971."
bad_summary = "This 1965 first edition is valued at 6800 USD, owned by one collector since 1985."

print("good_summary grounded?", numbers_in_summary_are_grounded(good_summary, source))
print("bad_summary grounded? ", numbers_in_summary_are_grounded(bad_summary, source))
assert numbers_in_summary_are_grounded(good_summary, source) is True
assert numbers_in_summary_are_grounded(bad_summary, source) is False
print("\nGrounding check correctly separates a faithful summary from a fabricated one.")

## Day 4: 경매 목록 테이블을 CSV 리포트로 스크래핑하기

희귀 도서 경매 사이트는 현재 목록을 HTML 테이블로 공개합니다.
BeautifulSoup으로 파싱하고, 행을 데이터프레임에 담고, 가격순으로
정렬하고, 결과를 정합성 검사한 뒤 CSV로 내보냅니다 — 이 과정 전체에
AI 모델은 전혀 필요하지 않습니다. HTML 테이블은 이미 구조화된
데이터이므로, 이것은 이해의 문제가 아니라 파싱의 문제입니다.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

# 가상의 페이지 HTML (실제로는 requests.get(...).text의 결과) -- 실제
# 페이지처럼 테이블 두 개(관련 없는 "관련 경매물" 위젯과 실제 목록
# 테이블)를 넣어서, find()가 올바른 하나를 정확히 골라내는지 보여준다.
sample_html = '''
<html><body>
<table class="related-lots"><tr><th>You might also like</th></tr></table>
<table class="listings">
  <tr><th>lot</th><th>title</th><th>current_bid</th></tr>
  <tr><td>101</td><td>Dune, 1965 1st ed.</td><td>1200</td></tr>
  <tr><td>102</td><td>Foundation, 1951 1st ed.</td><td>950</td></tr>
  <tr><td>103</td><td>I, Robot, 1950 1st ed.</td><td>800</td></tr>
</table>
</body></html>
'''

soup = BeautifulSoup(sample_html, "html.parser")
# find()는 첫 번째 매치만 반환한다; class로 필터링하기 때문에 관련 없는
# "related-lots" 테이블을 올바르게 건너뛸 수 있다
listings_table = soup.find("table", class_="listings")
rows = listings_table.find_all("tr")  # -> list[Tag], 길이 4 (헤더 1 + 데이터 3)
print("rows found (incl. header):", len(rows))

records = []
for row in rows[1:]:  # 헤더 행 건너뛰기
    cells_ = [td.get_text(strip=True) for td in row.find_all("td")]
    # -> 행당 길이 3의 list[str]: [lot, title, current_bid]
    records.append({
        "lot": int(cells_[0]),
        "title": cells_[1],
        "current_bid": float(cells_[2]),
    })

df = pd.DataFrame(records)
# -> DataFrame, 컬럼 [lot: int64, title: object, current_bid: float64]
print(df)
print(df.dtypes)
assert len(df) == 3
assert list(df.columns) == ["lot", "title", "current_bid"]

In [ ]:
def sanity_check_report(df: pd.DataFrame, expected_min_rows: int = 1) -> list[str]:
    """스크래핑한 데이터프레임을 리포트로 내보내기 전에 거치는 저렴한
    구조적 검사다 -- 스크래핑이 "성공"했지만(예외 없음) 페이지 레이아웃이
    바뀌어 데이터가 엉망이 된 경우를 잡아낸다.
    """
    problems = []
    if len(df) < expected_min_rows:
        problems.append(f"only {len(df)} rows, expected >= {expected_min_rows}")
    if df["current_bid"].isna().any():
        problems.append("current_bid has missing values")
    if (df["current_bid"] <= 0).any():
        problems.append("current_bid has non-positive values")
    if df["lot"].duplicated().any():
        problems.append("duplicate lot numbers (possible double-parsed table)")
    return problems  # -> list[str], 비어 있으면 "문제없음"

problems = sanity_check_report(df)
print("sanity check problems:", problems)
assert problems == []

report = df.sort_values("current_bid", ascending=False)
out_path = "auction_listings_report.csv"
report.to_csv(out_path, index=False)

with open(out_path) as f:
    written_csv = f.read()
print("\nwritten CSV:")
print(written_csv)
assert written_csv.splitlines()[1].startswith("101")  # 최고가(1200)가 먼저 정렬됨

In [ ]:
from urllib.robotparser import RobotFileParser

# 실제 사이트에 접근하지 않고도 robots.txt 검사의 동작 방식을 보여주기
# 위해, 네트워크 호출 없이 문자열 리터럴에서 파싱한다.
robots_txt = '''
User-agent: *
Disallow: /internal-notes/
Allow: /listings/
'''
rp = RobotFileParser()
rp.parse(robots_txt.strip().splitlines())
print("can fetch /listings/rare-books ?", rp.can_fetch("*", "/listings/rare-books"))
print("can fetch /internal-notes/bids ?", rp.can_fetch("*", "/internal-notes/bids"))
assert rp.can_fetch("*", "/listings/rare-books") is True
assert rp.can_fetch("*", "/internal-notes/bids") is False

# 배치 오류 처리: URL 하나가 "실패"해도(시뮬레이션) 배치 전체가 멈추지 않는다
def scrape_listing_page(url: str) -> dict:
    if "broken" in url:
        raise ValueError("no <table class='listings'> found on page")
    return {"url": url, "lots_found": 3}

auction_urls = [
    "https://example-auction.test/listings/rare-books",
    "https://example-auction.test/listings/broken-page",
    "https://example-auction.test/listings/first-editions",
]
results, failures = [], []
for url in auction_urls:
    try:
        results.append(scrape_listing_page(url))
    except Exception as exc:
        failures.append({"url": url, "error": str(exc)})

print(f"\n{len(results)} succeeded, {len(failures)} failed")
print("failures:", failures)
assert len(results) == 2 and len(failures) == 1

## 정리

이번 주 매일, 같은 원칙이 다른 형태로 반복됩니다. 먼저 mock/결정론적
경로를 만들고(Day 1), 모델 출력의 형태를 검증하지 않고는 신뢰하지
않으며(Day 2), 요약 속 숫자를 원문에서 근거를 확인하지 않고는 신뢰하지
않고(Day 3), 스크래핑 결과를 정합성 검사하지 않고는 성공했다고 믿지
않습니다(Day 4). 데이터의 형태에 따라 구체적인 검사(스키마 검증, 숫자
근거 검증, 데이터프레임 정합성 검사)는 달라지지만, 그 밑에 깔린 습관은
네 가지 서로 다른 실패 유형에 반복해서 적용되는 동일한 하나입니다.